In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv


# 🤖 Autonomous Data Science Agent — Interactive Evaluation Sandbox

Welcome Judges! This notebook provides a fully self-contained, live runtime environment for the Autonomous Data Science Agent. It automatically checks data integrity, sandboxes dynamic code generation, and executes real-time pipeline iterations using **Google Gemini 2.5 Flash**.

### 🚀 Quick-Start Instructions for Evaluation:
1. **Add Your Gemini API Key:** On the right-hand panel of this notebook, expand **Add-ons** ➡️ **Secrets**. Add a secret with the Label Name exactly as `GEMINI_API_KEY` and paste your valid Google AI Studio API key. Ensure the toggle switch for notebook access is turned **ON**.
2. **Set Your Target Variable:** If you wish to test a completely different prediction goal, simply update the `CHOSEN_TARGET_COLUMN` variable string in the execution cell below to match any column inside the dataset.
3. **Execute:** Click **Run All** in the notebook toolbar. Watch the agent loop dynamically construct state updates, handle feature expansions, and track convergence completely unassisted!

In [16]:
import os
import glob

# FORCE KAGGLE TO SYNC THE NEW DISK-PERSISTENCE FIX FROM GITHUB
print("🔄 Fetching the latest patched code from GitHub...")
%cd /kaggle/working

repo_dir = "/kaggle/working/autonomous-ds-agent"
repo_url = "https://github.com/25f3000007-code/autonomous-ds-agent.git"

if os.path.exists(repo_dir):
    print("📂 Repository found locally. Syncing latest changes...")
    %cd {repo_dir}
    !git reset --hard HEAD
    !git pull origin main
else:
    print("📥 Repository not found. Initializing fresh clone from GitHub...")
    !git clone {repo_url}
    %cd {repo_dir}

print("✅ Workspace successfully synchronized with your latest GitHub commit!")
print("-" * 60)


# List files to see the exact name of your cloned folder
print(os.listdir('/kaggle/working/'))

# Change directory into your repository (replace with your exact folder name)
os.chdir('/kaggle/working/autonomous-ds-agent')
print("Current Working Directory:", os.getcwd())

🔄 Fetching the latest patched code from GitHub...
/kaggle/working
📂 Repository found locally. Syncing latest changes...
/kaggle/working/autonomous-ds-agent
HEAD is now at c9e13b5 Add a pause to prevent rate limit errors with the AI
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.93 KiB | 1001.00 KiB/s, done.
From https://github.com/25f3000007-code/autonomous-ds-agent
 * branch            main       -> FETCH_HEAD
   c9e13b5..5eada6d  main       -> origin/main
Updating c9e13b5..5eada6d
Fast-forward
 ...ngineer-Your-task-is-to-refac_1782212988309.txt | 48 ++++++++++++++++++
 src/brain.py                                       | 57 ++++++++++++++--------
 2 files changed, 84 insertions(+), 21 deletions(-)
 create mode 100644 attached_assets/Pasted-You-are-an-expert-system-engineer-Your-task-is-to-refac_1782

In [14]:
!pip install -r requirements.txt

In [17]:
import time
import uuid
import sys
import subprocess
import pandas as pd
from kaggle_secrets import UserSecretsClient

# 🎯 STEP 1: CHOOSE YOUR TARGET COLUMN 
CHOSEN_TARGET_COLUMN = "SalePrice" 

print("🚀 INITIATING AUTOMATED ARTIFACT PIPELINE")
print("=" * 60)

# 🔄 STEP 2: FORCE KAGGLE TO SYNC THE NEW DISK-PERSISTENCE FIX FROM GITHUB
print("🔄 Fetching the latest patched code from GitHub...")
%cd /kaggle/working

repo_dir = "/kaggle/working/autonomous-ds-agent"
repo_url = "https://github.com/25f3000007-code/autonomous-ds-agent.git"

if os.path.exists(repo_dir):
    print("📂 Repository found locally. Syncing latest changes...")
    %cd {repo_dir}
    !git reset --hard HEAD
    !git pull origin main
else:
    print("📥 Repository not found. Initializing fresh clone from GitHub...")
    !git clone {repo_url}
    %cd {repo_dir}

print("✅ Workspace successfully synchronized with your latest GitHub commit!")
print("-" * 60)

# 🔑 STEP 3: SETUP ENVIRONMENT KEYS
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")

# 📂 STEP 4: LOCATE DATASET IN KAGGLE INPUT FOLDER
input_csv_files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
if not input_csv_files:
    raise FileNotFoundError("❌ Error: No CSV file found in your Kaggle Input section.")

kaggle_input_path = input_csv_files[0]
print(f"📂 Found input dataset at: {kaggle_input_path}")

# 🗺️ STEP 5: READ AND PREPARE MATRIX SCHEMA MAPPING FOR MAIN.PY
df = pd.read_csv(kaggle_input_path)
if CHOSEN_TARGET_COLUMN in df.columns:
    # Safely swap target names to match agent's internal expectations
    if 'price' in df.columns and CHOSEN_TARGET_COLUMN != 'price':
        df.rename(columns={'price': 'original_price_col'}, inplace=True)
        
    df.rename(columns={CHOSEN_TARGET_COLUMN: 'price'}, inplace=True)
    
    os.makedirs('data', exist_ok=True)
    local_target_path = 'data/uploaded_dataset.csv'
    df.to_csv(local_target_path, index=False)
    
    print(f"✅ Mapping setup complete! Target '{CHOSEN_TARGET_COLUMN}' ➡️ 'price'")
    print("-" * 60)
    
    # 🤖 STEP 6: KICK OFF THE AGENT FRAMEWORK WITH LIVE LOG CAPTURE
    print("🚀 Booting up Autonomous Data Science Agent Loop...")
    log_file_path = "/kaggle/working/trial_execution.log"
    cmd = "PYTHONPATH=. python src/main.py --dataset data/uploaded_dataset.csv --target price"
    
    with open(log_file_path, "w") as log_file:
        process = subprocess.Popen(
            cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        # Split outputs in real-time to both terminal console and execution log file
        for line in process.stdout:
            sys.stdout.write(line)
            log_file.write(line)
            log_file.flush()
            
    process.wait()
    print(f"\n📝 Trial log successfully written to: {log_file_path}")
    print("-" * 60)
    print("✨ Post-Processing: Preparing uniquely versioned deployment file...")

    # 💾 STEP 7: EXTRACT COMPLETED PHYSICAL ARTIFACT AFTER ITERATIONS COMPLETE
    if os.path.exists(local_target_path):
        df_optimized = pd.read_csv(local_target_path)
        
        # 🔄 STEP 8: REVERT TARGET LABELS BACK TO ORIGINAL FORM FOR CLEAN API USE
        df_optimized.rename(columns={'price': CHOSEN_TARGET_COLUMN}, inplace=True)
        if 'original_price_col' in df_optimized.columns:
            df_optimized.rename(columns={'original_price_col': 'price'}, inplace=True)
            
        # 🎲 STEP 9: GENERATE UNIQUE RANDOM FILENAME (ARTIFACT VERSIONING)
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        unique_id = uuid.uuid4().hex[:6]
        standalone_filename = f'optimized_deployment_{timestamp}_{unique_id}.csv'
        final_output_path = f'/kaggle/working/{standalone_filename}'
        
        # 💾 STEP 10: SAVE THE NEW DATA AS AN INDEPENDENT VERSION
        df_optimized.to_csv(final_output_path, index=False)
        
        # 🎯 COMPETITION ROUTING COPY
        competition_path = '/kaggle/working/submission.csv'
        df_optimized.to_csv(competition_path, index=False)
        
        # 🌟 FORCE KAGGLE ENGINE TO REGISTER EXPLICIT DIRECT NOTEBOOK WRITES
        print("\n📁 Syncing and registering files for Output panel and Kaggle CLI...")
        %cd /kaggle/working
        !cp {final_output_path} /kaggle/working/{standalone_filename}
        
        print(f"🚀 GENERATED EXPERIMENT ARTIFACT: {final_output_path}")
        print(f"🎯 GENERATED COMPETITION ARTIFACT: {competition_path}")
        print(f"📊 Transformed Features/Shape: {df_optimized.shape}")
        
        print("\n📊 Current Registered Files Ready for CLI Download:")
        print(os.listdir('/kaggle/working'))
        print("=" * 60)
    else:
        print("⚠️ Error: Target file was not updated by the agent script.")
else:
    print(f"❌ Target column '{CHOSEN_TARGET_COLUMN}' not found in the input dataset.")

🚀 INITIATING AUTOMATED ARTIFACT PIPELINE
🔄 Fetching the latest patched code from GitHub...
/kaggle/working
📂 Repository found locally. Syncing latest changes...
/kaggle/working/autonomous-ds-agent
HEAD is now at 5eada6d Add retry logic for API calls to prevent data processing errors
From https://github.com/25f3000007-code/autonomous-ds-agent
 * branch            main       -> FETCH_HEAD
Already up to date.
✅ Workspace successfully synchronized with your latest GitHub commit!
------------------------------------------------------------
📂 Found input dataset at: /kaggle/input/datasets/shashanknecrothapa/ames-housing-dataset/AmesHousing.csv
✅ Mapping setup complete! Target 'SalePrice' ➡️ 'price'
------------------------------------------------------------
🚀 Booting up Autonomous Data Science Agent Loop...

STARTING AUTONOMOUS DATA SCIENCE AGENT
✅ [Monitor] Successfully loaded dataset: /kaggle/working/autonomous-ds-agent/data/uploaded_dataset.csv
📊 [Agent] Baseline RMSE (Lower is better): 